# Lesson 4 — `einsum` & Tensor Contractions

## 学习目标

这一节的目标不是背诵 `torch.einsum()` 语法，而是建立一种新的 Tensor 思考方式：

> 用字母明确表示每一个维度，并直接描述哪些维度保留、哪些维度进行 contraction。

完成本节后，应能够：

1. 理解 Einstein Summation Convention；
2. 看懂 `torch.einsum()` 的基本 notation；
3. 使用 einsum 表示 dot product；
4. 使用 einsum 表示 matrix multiplication；
5. 使用 einsum 表示 batched matrix multiplication；
6. 使用 einsum 表示 Linear Projection；
7. 使用 einsum 推导 Self-Attention 中的 $QK^\top$；
8. 使用 einsum 推导 Attention Weights 与 $V$ 的乘法；
9. 理解 einsum 和 `@` / `matmul` 的关系；
10. 看到 `"bhtd,bhsd->bhts"` 时能够直接解释每个维度的含义。


## 1. 为什么需要 Einsum？

前面我们已经学会：

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

以及：

$$
(B,H,T,D_h)
(B,H,D_h,T)
\rightarrow
(B,H,T,T)
$$

这种 Shape Thinking 已经可以帮助我们推导矩阵乘法。

但是当 Tensor 维度越来越多时：

- Batch
- Sequence
- Head
- Head Dimension
- Expert
- Group
- Vocabulary

仅仅依赖 dim 0、dim 1、dim 2 很容易混乱。

einsum 提供了一种更接近数学符号的表达方式。

例如：

`torch.einsum("mk,kn->mn", A, B)`

直接表达：

$$
A_{mk}B_{kn}
\rightarrow
C_{mn}
$$

其中：

- `m` 保留；
- `n` 保留；
- `k` 出现在输入但没有出现在输出，因此沿 `k` 求和。

所以 einsum 的核心不是新的数学。

它只是把：

> 哪些维度对应，哪些维度求和

写得更加明确。


## 2. Einstein Summation Convention

Einstein Summation Convention 的核心思想是：

如果一个 index 在乘积中重复出现，并且没有出现在最终输出中，那么默认沿这个 index 求和。

例如：

$$
c_i
=
\sum_j A_{ij}x_j
$$

可以简写为：

$$
c_i=A_{ij}x_j
$$

这里：

- $i$ 出现在最终结果中，所以保留；
- $j$ 在右边重复出现，但结果中没有 $j$，因此沿 $j$ 求和。

PyTorch 的 einsum 将这个思想写成：

`"ij,j->i"`

其中：

- `ij` 表示第一个 Tensor 的两个维度；
- `j` 表示第二个 Tensor 的维度；
- `->i` 表示输出只保留 `i`。

所以：

$$
(i,j)(j)
\rightarrow
(i)
$$


## 3. `torch.einsum()` 的基本结构

基本形式：

`torch.einsum("input1,input2->output", tensor1, tensor2)`

例如：

`torch.einsum("ij,jk->ik", A, B)`

可以拆成：

### 第一个输入

`ij`

表示：

$$
A.shape=(i,j)
$$

### 第二个输入

`jk`

表示：

$$
B.shape=(j,k)
$$

### 输出

`ik`

表示：

$$
C.shape=(i,k)
$$

其中：

`j`

同时出现在两个输入中，但没有出现在输出中。

因此：

> 对 j 进行 contraction / summation。

也就是：

$$
C_{ik}
=
\sum_j
A_{ij}B_{jk}
$$


## 4. Dot Product

假设：

$$
a.shape=(D,)
$$

$$
b.shape=(D,)
$$

dot product：

$$
a\cdot b
=
\sum_d a_db_d
$$

einsum：

`"d,d->"`

注意输出箭头后面没有任何 index。

这表示所有维度都被 contraction。

因此输出是一个 scalar：

$$
(D)(D)
\rightarrow
()
$$


In [ ]:
import torch

a = torch.randn(4)
b = torch.randn(4)

result_einsum = torch.einsum("d,d->", a, b)

result_dot = torch.dot(a, b)

print("a:", a.shape)
print("b:", b.shape)

print("einsum:", result_einsum)
print("dot   :", result_dot)

print("same:", torch.allclose(result_einsum, result_dot))


a: torch.Size([4])
b: torch.Size([4])
einsum: tensor(-2.0908)
dot   : tensor(-2.0908)
same: True


## 5. Matrix-Vector Multiplication

假设：

$$
A.shape=(M,D)
$$

$$
x.shape=(D,)
$$

矩阵向量乘法：

$$
y=Ax
$$

shape：

$$
(M,D)(D)
\rightarrow
(M)
$$

einsum：

`"md,d->m"`

逐个字母分析：

- `m`：只存在于 A，并保留在输出；
- `d`：同时存在于 A 和 x；
- `d` 不出现在输出中，所以 contraction。

数学形式：

$$
y_m
=
\sum_d
A_{md}x_d
$$


In [ ]:
M = 3
D = 4

A = torch.randn(M, D)
x = torch.randn(D)

y_einsum = torch.einsum("md,d->m", A, x)

y_matmul = A @ x

print("A:", A.shape)
print("x:", x.shape)

print("einsum:", y_einsum.shape)
print("matmul:", y_matmul.shape)

print("same:", torch.allclose(y_einsum, y_matmul))


A: torch.Size([3, 4])
x: torch.Size([4])
einsum: torch.Size([3])
matmul: torch.Size([3])
same: True


## 6. Matrix Multiplication

上一节我们学过：

$$
A.shape=(M,K)
$$

$$
B.shape=(K,N)
$$

于是：

$$
AB.shape=(M,N)
$$

einsum 可以写成：

`"mk,kn->mn"`

也就是：

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

逐个 index 分析：

- `m`：保留；
- `n`：保留；
- `k`：两个输入都有，但输出没有，所以 contraction。

因此：

$$
C_{mn}
=
\sum_k
A_{mk}B_{kn}
$$

这与：

`A @ B`

完全是同一个数学运算。


In [3]:
M = 3
K = 4
N = 5

A = torch.randn(M, K)
B = torch.randn(K, N)

C_einsum = torch.einsum("mk,kn->mn", A, B)

C_matmul = A @ B

print("A:", A.shape)
print("B:", B.shape)

print("einsum:", C_einsum.shape)
print("matmul:", C_matmul.shape)

print("same:", torch.allclose(C_einsum, C_matmul))


A: torch.Size([3, 4])
B: torch.Size([4, 5])
einsum: torch.Size([3, 5])
matmul: torch.Size([3, 5])
same: True


## 7. Einsum 的核心阅读方法

以后看到 einsum，不要试图整句背下来。

固定做三件事。

### Step 1：给每个字母赋予语义

例如：

`btd`

可以读成：

$$
(B,T,D)
$$

其中：

- `b` = batch
- `t` = token / sequence
- `d` = model dimension

### Step 2：看哪些字母出现在输出中

输出中存在的 index：

> 保留。

### Step 3：看哪些输入 index 没有出现在输出中

输入中存在，但输出中消失的 index：

> contraction / summation。

例如：

`"btd,dh->bth"`

分析：

输入：

$$
(B,T,D)
$$

和：

$$
(D,H)
$$

输出：

$$
(B,T,H)
$$

所以：

- `b`：保留；
- `t`：保留；
- `h`：保留；
- `d`：消失，因此 contraction。

这就是 Linear Projection。


## 8. Linear Projection

Transformer 中：

$$
X.shape=(B,T,D)
$$

$$
W.shape=(D,H)
$$

上一节我们写：

`X @ W`

得到：

$$
(B,T,H)
$$

einsum 可以写：

`"btd,dh->bth"`

对应：

$$
X_{btd}W_{dh}
\rightarrow
Y_{bth}
$$

数学形式：

$$
Y_{bth}
=
\sum_d
X_{btd}W_{dh}
$$

其中只有：`d`被 contraction。

所以：

$$
(B,T,D)(D,H)
\rightarrow
(B,T,H)
$$


In [4]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)
weight = torch.randn(D, H)

y_einsum = torch.einsum("btd,dh->bth", x, weight)

y_matmul = x @ weight

print("x     :", x.shape)
print("weight:", weight.shape)

print("einsum:", y_einsum.shape)
print("matmul:", y_matmul.shape)

print("same:", torch.allclose(y_einsum, y_matmul))


x     : torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
einsum: torch.Size([2, 3, 6])
matmul: torch.Size([2, 3, 6])
same: True


## 9. Batched Matrix Multiplication

假设：

$$
A.shape=(B,M,K)
$$

$$
B.shape=(B,K,N)
$$

我们希望每个 batch 独立进行：

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

最终：

$$
(B,M,N)
$$

einsum：

`"bmk,bkn->bmn"`

分析：

- `b`：两个输入都有，同时输出也有，所以保留；
- `m`：保留；
- `n`：保留；
- `k`：两个输入都有，但输出没有，所以 contraction。

因此：

$$
(B,M,K)(B,K,N)
\rightarrow
(B,M,N)
$$


In [5]:
B = 2
M = 3
K = 4
N = 5

A = torch.randn(B, M, K)
B_tensor = torch.randn(B, K, N)

C_einsum = torch.einsum("bmk,bkn->bmn", A, B_tensor)

C_matmul = A @ B_tensor

print("A:", A.shape)
print("B:", B_tensor.shape)

print("einsum:", C_einsum.shape)
print("matmul:", C_matmul.shape)

print("same:", torch.allclose(C_einsum, C_matmul))


A: torch.Size([2, 3, 4])
B: torch.Size([2, 4, 5])
einsum: torch.Size([2, 3, 5])
matmul: torch.Size([2, 3, 5])
same: True


## 10. Multi-Head Attention 的 Index Naming

从现在开始，我们统一使用：

$$
b=\text{batch}
$$

$$
h=\text{attention head}
$$

$$
t=\text{query token}
$$

$$
s=\text{key / source token}
$$

$$
d=\text{head dimension}
$$

特别注意：

在 Attention 中，我们经常需要两个 sequence index。

因此不应该把 Query token 和 Key token 都写成 `t`。

更加清晰的写法是：

- `t`：Query Position
- `s`：Key / Source Position

例如：

$$
Q.shape=(B,H,T,D_h)
$$

einsum notation：

`bhtd`

而：

$$
K.shape=(B,H,S,D_h)
$$

notation：

`bhsd`

在 Self-Attention 中：

$$
T=S
$$

但我们仍然使用两个不同字母。

这样才能清楚表达：

> Query position 和 Key position 是两个不同的轴。


## 11. Attention：$QK^T$

假设：

$$
Q.shape=(B,H,T,D_h)
$$

记为：

`bhtd`

而：

$$
K.shape=(B,H,S,D_h)
$$

记为：

`bhsd`

我们希望计算：

> 每一个 Query token 和每一个 Key token 的 dot product。

因此：

$$
d
$$

应该被 contraction。

但是：

- Batch `b` 保留；
- Head `h` 保留；
- Query Position `t` 保留；
- Key Position `s` 保留。

所以：

`"bhtd,bhsd->bhts"`

也就是：

$$
Q_{bhtd}K_{bhsd}
\rightarrow
S_{bhts}
$$

其中：

$$
S_{bhts}
=
\sum_d
Q_{bhtd}K_{bhsd}
$$

最终：

$$
(B,H,T,D_h)
(B,H,S,D_h)
\rightarrow
(B,H,T,S)
$$

Self-Attention 中：

$$
S=T
$$

所以：

$$
(B,H,T,T)
$$


In [6]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)

scores_einsum = torch.einsum("bhtd,bhsd->bhts", q, k)

scores_matmul = q @ k.transpose(-2, -1)

print("Q:", q.shape)
print("K:", k.shape)

print("einsum:", scores_einsum.shape)

print("matmul:", scores_matmul.shape)

print("same:", torch.allclose(scores_einsum, scores_matmul))


Q: torch.Size([2, 4, 8, 16])
K: torch.Size([2, 4, 8, 16])
einsum: torch.Size([2, 4, 8, 8])
matmul: torch.Size([2, 4, 8, 8])
same: True


## 12. 为什么 Einsum 中没有 `K.transpose()`？

上一节使用 Matrix Multiplication 时：

$$
Q.shape=(B,H,T,D_h)
$$

$$
K.shape=(B,H,T,D_h)
$$

必须先：

`K.transpose(-2, -1)`

得到：

$$
(B,H,D_h,T)
$$

然后：

$$
QK^\top
$$

但是 einsum：

`"bhtd,bhsd->bhts"`

不需要显式 transpose。

因为我们已经通过 index notation 明确告诉 PyTorch：

- `d` 是 contraction dimension；
- `t` 是 Query Position；
- `s` 是 Key Position。

einsum 不要求 contraction dimension 一定物理出现在“倒数第一维和倒数第二维”。

它直接根据 index 名称确定如何组合 Tensor。

所以：

`q @ k.transpose(-2, -1)`

和：

`torch.einsum("bhtd,bhsd->bhts", q, k)`

表达的是同一个数学运算。


## 13. `bhts` 的语义

Attention Score：

$$
scores.shape=(B,H,T,S)
$$

einsum notation：

`bhts`

逐个维度：

- `b`：Batch
- `h`：Attention Head
- `t`：Query Position
- `s`：Key Position

因此：

$$
scores[b,h,t,s]
$$

表示：

> batch b 中，第 h 个 Attention Head，第 t 个 Query token 对第 s 个 Key token 的 score。

这比只写：

$$
(B,H,T,T)
$$

更进一步。

Shape 只告诉我们大小。

einsum 的 index notation 同时告诉我们：

> 每个维度的语义。


## 14. Attention Weights 与 V

Softmax 后：

$$
A.shape=(B,H,T,S)
$$

记作：

`bhts`

Value：

$$
V.shape=(B,H,S,D_h)
$$

记作：

`bhsd`

我们希望对所有 source token：

$$
s
$$

做 weighted sum。

因此 `s` 应该被 contraction。

输出保留：

- `b`
- `h`
- `t`
- `d`

所以 einsum：

`"bhts,bhsd->bhtd"`

即：

$$
A_{bhts}V_{bhsd}
\rightarrow
O_{bhtd}
$$

数学形式：

$$
O_{bhtd}
=
\sum_s
A_{bhts}V_{bhsd}
$$

最终：

$$
(B,H,T,S)
(B,H,S,D_h)
\rightarrow
(B,H,T,D_h)
$$


In [7]:
B = 2
H = 4
T = 8
Dh = 16

attention = torch.randn(B, H, T, T)

v = torch.randn(B, H, T, Dh)

output_einsum = torch.einsum("bhts,bhsd->bhtd", attention, v)

output_matmul = attention @ v

print("attention:", attention.shape)

print("V:", v.shape)

print("einsum:", output_einsum.shape)

print("matmul:", output_matmul.shape)

print("same:", torch.allclose(output_einsum, output_matmul))


attention: torch.Size([2, 4, 8, 8])
V: torch.Size([2, 4, 8, 16])
einsum: torch.Size([2, 4, 8, 16])
matmul: torch.Size([2, 4, 8, 16])
same: True


## 15. Attention 的 Einsum 表达

Self-Attention 最核心的两个矩阵乘法，现在可以写成：

### Query-Key Similarity

$$
QK^\top
$$

einsum：

`"bhtd,bhsd->bhts"`

得到：

$$
(B,H,T,S)
$$

### Weighted Value Sum

$$
AV
$$

einsum：

`"bhts,bhsd->bhtd"`

得到：

$$
(B,H,T,D_h)
$$

所以核心流程：

$$
Q,K
$$

$$
\downarrow
$$

$$
scores_{bhts}
=
\sum_d
Q_{bhtd}K_{bhsd}
$$

$$
\downarrow
$$

$$
A=\operatorname{softmax}(scores)
$$

$$
\downarrow
$$

$$
O_{bhtd}
=
\sum_s
A_{bhts}V_{bhsd}
$$

这里出现了两种不同的 contraction：

第一步 contraction：

$$
d
$$

第二步 contraction：

$$
s
$$

理解这一点非常重要。


In [8]:
import math

B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)


scores = torch.einsum("bhtd,bhsd->bhts", q, k)

scores = scores / math.sqrt(Dh)

attention = torch.softmax(scores, dim=-1)

output = torch.einsum("bhts,bhsd->bhtd", attention, v)


print("Q        :", q.shape)
print("K        :", k.shape)
print("V        :", v.shape)

print("Scores   :", scores.shape)
print("Attention:", attention.shape)
print("Output   :", output.shape)


Q        : torch.Size([2, 4, 8, 16])
K        : torch.Size([2, 4, 8, 16])
V        : torch.Size([2, 4, 8, 16])
Scores   : torch.Size([2, 4, 8, 8])
Attention: torch.Size([2, 4, 8, 8])
Output   : torch.Size([2, 4, 8, 16])


## 16. 什么情况下 Index 会被 Contraction？

考虑：

`"bhtd,bhsd->bhts"`

两个输入都包含：

- `b`
- `h`
- `d`

但是只有：

$$
d
$$

被 contraction。

为什么？

因为：

- `b` 仍然出现在输出中；
- `h` 仍然出现在输出中；
- `d` 没有出现在输出中。

所以判断 contraction 的真正规则是：

> 一个 index 出现在输入中，但是没有出现在输出中。

不能简单理解成：

> 只要一个字母在两个输入里重复就一定求和。

例如：

`b`

同时出现在两个输入里，但输出也有 `b`。

所以 Batch 维只是对应匹配，不会被求和。


## 17. Einsum 与 Reduction

einsum 不仅可以表示矩阵乘法，也可以表示 reduction。

例如：

$$
x.shape=(B,T,D)
$$

如果写：

`"btd->bt"`

那么：

- `b` 保留；
- `t` 保留；
- `d` 消失。

因此沿：

$$
D
$$

求和。

结果：

$$
(B,T)
$$

也就是：

$$
y_{bt}
=
\sum_d x_{btd}
$$

这等价于：

`x.sum(dim=-1)`

注意：

einsum 默认这里做的是求和，不是 mean。

如果需要 mean，还需要除以 $D$。


In [9]:
B = 2
T = 3
D = 4

x = torch.randn(B, T, D)

sum_einsum = torch.einsum("btd->bt", x)

sum_torch = x.sum(dim=-1)

print("einsum:", sum_einsum.shape)

print("torch:", sum_torch.shape)

print("same:", torch.allclose(sum_einsum, sum_torch))


einsum: torch.Size([2, 3])
torch: torch.Size([2, 3])
same: True


## 18. Outer Product

einsum 还可以非常直观地表达 Outer Product。

假设：

$$
a.shape=(M,)
$$

$$
b.shape=(N,)
$$

如果写：

`"m,n->mn"`

注意：

输入中没有任何重复 index。

因此：

> 没有任何 contraction。

结果：

$$
(M,N)
$$

数学形式：

$$
C_{mn}=a_mb_n
$$

这就是 Outer Product。

所以可以形成一个很重要的直觉：

### 有 index 消失

意味着 contraction / reduction。

### 所有 index 都保留

通常意味着维度组合，例如 outer product。


In [10]:
a = torch.randn(3)
b = torch.randn(4)

outer_einsum = torch.einsum("m,n->mn", a, b)

outer_torch = torch.outer(a, b)

print("einsum:", outer_einsum.shape)

print("torch:", outer_torch.shape)

print("same:", torch.allclose(outer_einsum, outer_torch))


einsum: torch.Size([3, 4])
torch: torch.Size([3, 4])
same: True


## 19. Einsum 与维度重排

考虑二维矩阵：

$$
A.shape=(M,N)
$$

如果写：

`"mn->nm"`

没有任何 index 被删除。

只是输出顺序从：

$$
(m,n)
$$

变成：

$$
(n,m)
$$

所以：

$$
(M,N)
\rightarrow
(N,M)
$$

这就是 transpose。

因此 einsum 甚至可以表达某些维度重排操作。

不过工程代码里单纯 transpose 时：

`tensor.transpose(...)`

或：

`tensor.permute(...)`

通常会更加直接。

einsum 的主要价值还是：

> 清晰表达 contraction。


In [11]:
A = torch.randn(3, 4)

B_einsum = torch.einsum("mn->nm", A)

B_transpose = A.transpose(0, 1)

print("A:", A.shape)

print("einsum:", B_einsum.shape)

print("transpose:", B_transpose.shape)

print("same:", torch.allclose(B_einsum, B_transpose))


A: torch.Size([3, 4])
einsum: torch.Size([4, 3])
transpose: torch.Size([4, 3])
same: True


## 20. 用一个统一视角理解 Einsum

目前我们已经看到 einsum 可以表达三类操作。

### 1. Reduction / Contraction

例如：

`"btd->bt"`

$$
(B,T,D)
\rightarrow
(B,T)
$$

这里：

$$
d
$$

消失。

---

### 2. Dimension Reordering

例如：

`"mn->nm"`

$$
(M,N)
\rightarrow
(N,M)
$$

没有维度消失，只改变顺序。

---

### 3. Dimension Combination

例如：

`"m,n->mn"`

$$
(M)(N)
\rightarrow
(M,N)
$$

没有 contraction，而是组合两个独立维度。

---

矩阵乘法实际上就是这些规则的组合：

`"mk,kn->mn"`

其中：

- `m` 保留；
- `n` 保留；
- `k` contraction。

所以 einsum 可以理解成：

> 一种非常通用的 Tensor index transformation language。


## 21. `einsum` vs `matmul`

能够用 einsum 表示一个运算，不代表正式代码里一定应该使用 einsum。

例如普通 Linear Projection：

`x @ weight`

通常比：

`torch.einsum("btd,dh->bth", x, weight)`

更加直接。

所以可以采用下面的原则。

### 优先使用 `@` / matmul

当运算明显就是标准矩阵乘法时：

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

例如：

- Linear Projection
- QK^T
- Attention × V

### einsum 很适合

当：

- Tensor 维度很多；
- contraction 关系不容易读；
- 论文公式天然使用 index notation；
- 需要多个不同维度之间进行组合；
- 学习阶段希望明确每个维度的语义。

因此在本课程中：

> einsum 首先是一种帮助理解 Tensor contraction 的工具，而不是为了把所有矩阵乘法都改写成 einsum。


## 22. 常见错误

### 错误 1：字母数量和 Tensor ndim 不一致

例如：

$$
x.shape=(B,T,D)
$$

是三维 Tensor。

那么：

`"btd"`

必须有三个 index。

不能写：

`"bt"`

---

### 错误 2：相同 index 的 size 不一致

例如：

`"btd,dh->bth"`

这里两个 Tensor 都使用 `d`。

因此两个 `d` 的 size 必须相同。

---

### 错误 3：忘记输出中的 index 顺序决定输出 shape

例如：

`"btd->bdt"`

和：

`"btd->btd"`

完全不同。

前者相当于交换 T 和 D。

---

### 错误 4：把重复 index 都理解为 contraction

真正规则是：

> 输入中存在，但是输出中没有出现的 index 才会被 contraction。

---

### 错误 5：Self-Attention 中把两个 token 轴都叫 `t`

为了表达 token-to-token relation，更推荐：

- `t` = Query Position
- `s` = Key / Source Position

所以：

`"bhtd,bhsd->bhts"`

比使用同一个 token 字母更容易理解。


## 本节总结

### Rule 1：Einsum 用字母表示 Tensor 维度

例如：

$$
(B,T,D)
$$

写成：

`btd`

---

### Rule 2：输出中存在的 Index 被保留

例如：

`"btd,dh->bth"`

输出：

`bth`

所以：

- b 保留；
- t 保留；
- h 保留。

---

### Rule 3：输入中存在但输出中消失的 Index 被 Contraction

例如：

`"btd,dh->bth"`

其中：

`d`

消失，因此：

$$
\sum_d
$$

---

### Rule 4：Matrix Multiplication

`"mk,kn->mn"`

等价于：

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

---

### Rule 5：Linear Projection

`"btd,dh->bth"`

表示：

$$
(B,T,D)(D,H)
\rightarrow
(B,T,H)
$$

---

### Rule 6：Attention QK^T

`"bhtd,bhsd->bhts"`

表示：

$$
(B,H,T,D_h)
(B,H,S,D_h)
\rightarrow
(B,H,T,S)
$$

其中 contraction dimension：

$$
D_h
$$

---

### Rule 7：Attention × V

`"bhts,bhsd->bhtd"`

表示：

$$
(B,H,T,S)
(B,H,S,D_h)
\rightarrow
(B,H,T,D_h)
$$

其中 contraction dimension：

$$
S
$$

---

### Rule 8：Einsum 的真正价值

einsum 不只是一个 PyTorch API。

它训练的是：

> Tensor index reasoning。

看到：

`bhtd`

应该逐渐直接想到：

$$
Batch
\times
Head
\times
Token
\times
HeadDimension
$$
